<a href="https://colab.research.google.com/github/cedo-cloud/plp-da-a3-duka-sales-and-agent-cleanup/blob/main/agent_cleanup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mobile Money Agent Network Cleanup

In [ ]:
import pandas as pd
import numpy as np
import os
from cleaning_pipeline import clean_agent_data

# Load raw data
txns_raw = pd.read_csv("agent_transactions.csv")
agents_raw = pd.read_csv("agents.csv")

# Data-Quality Audit Table
audit_data = {
    "issue": [
        "Duplicate txn_id values",
        "Inconsistent txn_type casing/whitespace",
        "Missing amounts (NaN)",
        "Invalid amounts ('N/A')",
        "Date format inconsistencies (slashes)",
        "Unknown agent_id (AG099)"
    ],
    "dimension": ["Uniqueness", "Consistency", "Completeness", "Validity", "Validity", "Referential Integrity"],
    "column": ["txn_id", "txn_type", "amount", "amount", "txn_date", "agent_id"],
    "rows affected": [
        txns_raw.duplicated(subset=['txn_id']).sum(),
        txns_raw['txn_type'].ne(txns_raw['txn_type'].str.strip().str.title()).sum(),
        txns_raw['amount'].isna().sum(),
        (txns_raw['amount'] == 'N/A').sum(),
        txns_raw['txn_date'].str.contains('/').sum(),
        (txns_raw['agent_id'] == 'AG099').sum()
    ]
}
audit_df = pd.DataFrame(audit_data)
display(audit_df)

,issue,dimension,column,rows affected
0,Duplicate txn_id values,Uniqueness,txn_id,12
1,Inconsistent txn_type casing/whitespace,Consistency,txn_type,384
2,Missing amounts (NaN),Completeness,amount,22
3,Invalid amounts ('N/A'),Validity,amount,0
4,Date format inconsistencies (slashes),Validity,txn_date,5
5,Unknown agent_id (AG099),Referential Integrity,agent_id,8


In [ ]:
# Run cleaning pipeline
clean_df, log = clean_agent_data(txns_raw, agents_raw)

print("--- Cleaning Log ---")
for entry in log:
    print(entry)

--- Cleaning Log ---
Uniqueness: Removed 12 duplicate transactions.
Completeness: Imputed 20 missing/invalid amounts with median (1776.5).
Outliers: Flagged 53 transactions above IQR upper bound (5514).
Merge: Found 7 transactions with unknown agents (e.g., AG099). Kept them with NaN county.


### Post-Merge Validation

In [ ]:
print(f"Original transaction rows: {len(txns_raw)}")
print(f"Clean transaction rows: {len(clean_df)}")
print(f"Transactions with unknown agents (NaN county): {clean_df['county'].isna().sum()}")

Original transaction rows: 612
Clean transaction rows: 600
Transactions with unknown agents (NaN county): 7


### Summarise: County and Weekly Pivots

In [ ]:
# County summary
county_summary = clean_df.groupby(["county", "txn_type"]).agg(
    total_value=("amount", "sum"),
    num_transactions=("txn_id", "count"),
    avg_value=("amount", "mean")
).round(2).reset_index()

print("County Summary:\n", county_summary)

# Weekly pivot table
clean_df["week"] = clean_df["txn_date"].dt.isocalendar().week
weekly_pivot = pd.pivot_table(clean_df, values="amount", index="county", columns="week", aggfunc="sum", fill_value=0)
print("\nWeekly Pivot Table:\n", weekly_pivot)

County Summary:
     county    txn_type  total_value  num_transactions  avg_value
0   Kiambu     Deposit     334737.0                43    7784.58
1   Kiambu  Withdrawal      77050.5                27    2853.72
2   Kisumu     Deposit     436420.5                74    5897.57
3   Kisumu  Withdrawal      76374.5                37    2064.18
4  Mombasa     Deposit     148986.0                72    2069.25
5  Mombasa  Withdrawal     368592.0                61    6042.49
6  Nairobi     Deposit     543590.0               118    4606.69
7  Nairobi  Withdrawal     162036.0                62    2613.48
8   Nakuru     Deposit     184890.0                60    3081.50
9   Nakuru  Withdrawal     328594.5                39    8425.50

Weekly Pivot Table:
 week         9         10        11       12       13        14        15  \
county                                                                      
Kiambu      0.0   18317.0  267771.5  22054.0  11504.0   22609.0   15030.0   
Kisumu   1893.0

In [ ]:
# Export clean data
os.makedirs("outputs", exist_ok=True)
clean_df.to_csv("outputs/agent_transactions_clean.csv", index=False)
print("Clean data exported to outputs/agent_transactions_clean.csv")

Clean data exported to outputs/agent_transactions_clean.csv


# Data Quality Report

**1. Uniqueness:** Removed duplicate transaction IDs to ensure each transaction is counted exactly once in the ledger.
**2. Consistency:** Standardized transaction types to exactly "Deposit" and "Withdrawal" by stripping whitespace and standardizing casing.
**3. Completeness:** Handled missing and "N/A" amounts by imputing them with the median transaction value. This prevents data loss while maintaining the statistical distribution. An `amount_imputed` flag was added to track these records.
**4. Validity:** Converted amounts to numeric and dates to ISO datetime format, correcting slash-separated dates to ensure proper time-series analysis.
**5. Outliers:** Flagged transactions exceeding the IQR upper bound. The KES 250,000 values were flagged as outliers. *Recommendation:* The business should investigate these high-value transactions to confirm if they are legitimate large transfers or system errors before including them in standard performance metrics.
**6. Referential Integrity:** Merged with the agent register. Transactions with unknown agents (e.g., AG099) were retained but left with a null county to preserve transaction volume metrics, while flagging them for master data management review.

### Privacy & Data Protection
If this extract contained customer phone numbers, these would be classified as Personal Identifiable Information (PII) under the **Kenya Data Protection Act 2019**. To protect this data, phone numbers must be encrypted at rest and in transit, access must be restricted to authorized personnel on a strict need-to-know basis, and they should be masked (e.g., `+2547***1234`) in any analytical outputs or shared reports to ensure compliance with data minimization and privacy principles.